# 05 · Backtest: six models, one year ahead

**Purpose.** Forecast every test year with each of the six models, one year ahead, at POE10, POE50 and POE90, and save the forecasts for notebooks 06 and 07.

**Paper section served.** III Method.

**Inputs**

- `data/interim/network_30min.parquet` from notebook 01.
- `data/interim/pv_daily.parquet` and `weather_daily.parquet` from notebook 02.

**Paper artifacts**

- `outputs/tables/table1.csv`: Table I, the six models.

**Supporting checks** (`outputs/checks`)

- `check_complete_days.csv`: complete days per year behind the robustness target.
- `check_m5_pv_range.csv`: plain XGBoost against the PV capacity it was trained on.
- `check_holiday_miss.csv`: the miss on public holidays against ordinary weekend days.
- `check_m4_terms.csv`: M4 with one group of weather terms switched off.
- `check_m4_coefficients.csv`: M4's weather coefficients, and how far PV x solar is extrapolated.

**Data for later notebooks** (`data/interim`)

- `forecasts.parquet`: one row per model, target, test year and month.
- `actuals.parquet`: the actual value of every target, with its validity.

**Main process**

1. Set up.
2. What is forecast: three targets.
3. The six models (Table I).
4. Run the rolling-origin backtest.
5. Check: plain XGBoost beyond the PV capacity it was trained on.
6. Check: does the single non-working-day flag hide a holiday effect?
7. Check: which weather terms pull M4's forecasts down?
8. Result.

Earlier looks at these forecasts that are not used by the paper (a first table of POE50 against the actuals, the day M3 predicts as lowest, the holiday miss by holiday) are in `notebooks/archive/A1_model_exploration.ipynb`.

**How to read POE.** POE is probability of exceedance. POE90 is the low side: demand is expected to stay above it nine years in ten. POE10 is the high side. POE50 is the middle.

**What is judged where.** H1 (M3 against M2) and H2 (M4 against M3) are judged on the **daytime low**, annual and monthly. The whole-day minimum is the headline forecast target and is reported alongside. This was fixed before any model was run.

## 1. Set up

In [1]:
import pandas as pd

from pynmdf import evaluate, models, targets
from pynmdf.config import load_config

cfg = load_config()
interim, tables, checks = (cfg["dirs"][key] for key in ("interim", "tables", "checks"))
wanted = cfg["targets"]
first_test, last_year = cfg["evaluation"]["first_test_year"], cfg["study"]["last_year"]

pd.set_option("display.width", 220, "display.max_columns", 30)
pd.set_option("display.float_format", "{:,.0f}".format)
total = pd.read_parquet(interim / "network_30min.parquet")["total_mw"]
pv = pd.read_parquet(interim / "pv_daily.parquet")["pv_mw"]
weather = pd.read_parquet(interim / "weather_daily.parquet")

## 2. What is forecast: three targets

Each target exists for every year and every month.

- **Whole-day minimum**: the lowest valid half hour. The headline target.
- **Daytime low**: the lowest valid half hour between 09:00 and 17:00. H1 and H2 are judged on it.
- **Robustness target**: the mean of the 15 lowest daily minima of the year (2 lowest in a month), over complete days only. It asks whether a result depends on one extreme half hour.

A year always counts. A month counts only when at least 90% of it is kept (90% of its daytime half hours, for the daytime low).

The first table gives the annual actuals, with the number of complete days each robustness value is drawn from. The second counts the monthly test points that can be scored, out of 84.

In [2]:
actuals = targets.period_targets(
    total, wanted["troughs"], wanted["month_min_kept_share"], wanted["robust_days"]
)
actuals.to_parquet(interim / "actuals.parquet")

annual = actuals[actuals["level"] == "annual"]
shown = annual.pivot(index="year", columns="target", values="actual")
shown["complete_days"] = annual[annual["target"] == "robust"].set_index("year")["complete_days"]
shown[["complete_days"]].to_csv(checks / "check_complete_days.csv")
shown.T

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
target,,,,,,,,,,,
daytime_low,"1,833","1,891","1,888","1,902","1,759","1,671","1,357","1,458","1,080","1,019",880
minimum,"1,368","1,433","1,474","1,495","1,473","1,429","1,357","1,428","1,080","1,019",880
robust,"1,403","1,448","1,508","1,538","1,525","1,461","1,408","1,467","1,218","1,120","1,024"
complete_days,357,361,360,365,353,347,331,352,342,309,359


In [3]:
in_test = actuals[(actuals["level"] == "monthly") & (actuals["year"] >= first_test)]
in_test.groupby("target")["valid"].agg(valid_months="sum", months="size").T

target,daytime_low,minimum,robust
valid_months,77,76,76
months,84,84,84


## 3. The six models (Table I)

M1 repeats last year. M2 to M6 are fitted on the same daily data: each predicts the day's overnight low and daytime low, and the day's minimum is the lower of the two. M2 to M5 are a ladder, each step adding one thing.

- **M2 Time trend** knows the calendar (the month, and whether the day is a working day) and time, with a separate trend for each month.
- **M3 PV only** is M2 with PV capacity in place of time.
- **M4 PV + weather** adds PV capacity times solar exposure, and the day's maximum and minimum temperature.
- **M5** is given the same information as M4, in XGBoost.
- **M6** is given the same information as M3 (month, non-working day, PV capacity), in XGBoost.

**M6 is supplementary.** It was added after the scores of M1 to M5 had been seen. M5 differs from M3 in two ways at once, weather and the form of the model, so its score could not say which of the two helped. M5 against M6 changes the weather only, in tree form. The two comparisons fixed in advance stay as they were: M3 against M2, and M4 against M3.

A **non-working day** is a Saturday, a Sunday or a NSW public holiday, as one flag. An earlier version used two flags, weekend and holiday. The linear models then added the two effects on a holiday that fell on a weekend and predicted it about 1,000 MW below a working day, far lower than such days are. Section 6 checks the single flag.

In [4]:
models.MODEL_INFO.to_csv(tables / "table1.csv", index=False)
models.MODEL_INFO

,ID,Model,Fitted on,PV,Weather,POE from
0,M1,Naive,Annual or monthly values,No,No,Past errors
1,M2,Time trend,Daily lows,No,No,Replay of past residuals
2,M3,PV only,Daily lows,Yes,No,Replay of past residuals
3,M4,"PV + weather, linear",Daily lows,Yes,Yes,Weather replay and residuals
4,M5,"PV + weather, XGBoost",Daily lows,Yes,Yes,Weather replay and residuals
5,M6,"PV only, XGBoost",Daily lows,Yes,No,Replay of past residuals


## 4. Run the rolling-origin backtest

Each test year is forecast from data up to the end of the year before: train on 2015 to 2018 and forecast 2019, then add a year and forecast 2020, and so on to 2025. PV capacity of the target year is taken as known.

**How a range is produced.** M1 adds to last year's value a quantile of its own past year-on-year changes. For M2 to M6 the target year is simulated many times. Each simulation takes the model's prediction for every day under one historical weather year, then adds each day's miss borrowed from one left-out training year: the same calendar month, and both troughs from the same day. A miss is "left-out" because it comes from refitting the model without that year, so the model had not seen it. The spread of the simulated lows gives POE10, POE50 and POE90.

M2, M3 and M6 know nothing about the weather, so every weather year gives them the same prediction and all of their spread comes from the borrowed misses.

This cell takes a few minutes: XGBoost is refitted once for every left-out year at every origin.

In [5]:
forecasts = evaluate.rolling_origin(total, pv, weather, cfg)
forecasts.to_parquet(interim / "forecasts.parquet")

print(f"{len(forecasts):,} forecasts: {forecasts['valid'].sum():,} can be scored")
forecasts.groupby(["target", "level", "model"])["valid"].sum().unstack("model")

1,638 forecasts: 1,500 can be scored


model                M1  M2  M3  M4  M5  M6
target      level                          
daytime_low annual    7   7   7   7   7   7
            monthly  77  77  77  77  77  77
minimum     annual    7   7   7   7   7   7
            monthly  76  76  76  76  76  76
robust      annual    7   7   7   7   7   7
            monthly  76  76  76  76  76  76

## 5. Check: plain XGBoost beyond the PV capacity it was trained on

A tree model cannot extrapolate: asked about a PV capacity higher than any it has seen, it answers as if capacity had stopped at the highest value in its training data. PV capacity grows every year, so every target year lies outside M5's training range.

The table shows, for each test year, the highest PV capacity in the training data and the capacity at mid-year of the target year, then the daytime-low forecasts of M4 (linear, which can extrapolate) and M5 beside the actual. `central` is the forecast with no borrowed miss added; POE50 includes them. A positive error means the forecast was too low.

In [6]:
daytime = forecasts[(forecasts["target"] == "daytime_low") & (forecasts["level"] == "annual")]
by_model = daytime.pivot(index="year", columns="model", values=["central", "poe50"])
actual = daytime.groupby("year")["actual"].first()

m5_check = pd.DataFrame(
    {
        "pv_max_in_training_mw": [pv[: f"{year - 1}"].max() for year in actual.index],
        "pv_target_mid_year_mw": [pv[f"{year}-06-30"] for year in actual.index],
        "actual_mw": actual,
        "m4_central": by_model["central", "M4"],
        "m5_central": by_model["central", "M5"],
        "m4_poe50": by_model["poe50", "M4"],
        "m5_poe50": by_model["poe50", "M5"],
    }
)
m5_check["m4_error"] = m5_check["actual_mw"] - m5_check["m4_poe50"]
m5_check["m5_error"] = m5_check["actual_mw"] - m5_check["m5_poe50"]
m5_check.to_csv(checks / "check_m5_pv_range.csv")
m5_check.T

year,2019,2020,2021,2022,2023,2024,2025
pv_max_in_training_mw,535,701,916,"1,172","1,428","1,723","1,984"
pv_target_mid_year_mw,620,780,"1,050","1,291","1,563","1,882","2,084"
actual_mw,"1,759","1,671","1,357","1,458","1,080","1,019",880
m4_central,"1,886","1,704","1,405","1,106","1,046",863,778
m5_central,"1,931","1,829","1,599","1,456","1,564","1,154","1,044"
m4_poe50,"1,721","1,576","1,307",999,921,757,683
m5_poe50,"1,810","1,680","1,410","1,293","1,332",994,881
m4_error,39,95,50,459,159,262,197
m5_error,-51,-9,-53,165,-252,25,-1


## 6. Check: does the single non-working-day flag hide a holiday effect?

The models treat a public holiday and an ordinary weekend day as the same kind of day. If public holidays were really lower than ordinary weekend days, the models would miss them on the low side: the actual would sit below the prediction, and the average miss on holidays would be clearly negative next to the average miss on ordinary weekend days.

For this check each model is fitted up to the origin and then given each day's **own** weather, with no replay and no borrowed miss. That is not a forecast (the weather is not known a year ahead); it shows what the fitted model says about each day once its inputs are known.

The table gives, for each low and each model, the average miss (actual minus predicted, MW) on public holidays and on ordinary weekend days over the seven test years, with the number of days behind each, and the difference between the two.

In [7]:
subdivision = cfg["calendar"]["holiday_subdivision"]
daily = evaluate.own_weather_predictions(total, pv, weather, cfg)

holiday_miss = evaluate.miss_on_holidays(daily, subdivision)
holiday_miss.to_csv(checks / "check_holiday_miss.csv")
holiday_miss

holiday_mean_miss_mw  holiday_days  weekend_mean_miss_mw  weekend_days  holiday_minus_weekend_mw
low       model                                                                                                  
daytime   M2                     -210            76                   -51           667                      -159
          M3                     -101            76                    50           667                      -150
          M4                      -80            76                    42           667                      -123
          M5                     -111            76                    16           667                      -126
          M6                     -110            76                   -13           667                       -96
overnight M2                      -85            76                   -40           667                       -45
          M3                      -85            76                   -40           667                       -44
          M4                      -83            76                   -41           667                       -42
          M5                      -27            76                    21           667                       -49
          M6                      -38            76                     5           667                       -43

## 7. Check: which weather terms pull M4's forecasts down?

M4's forecasts of the annual low sit below M3's, and below the actual in most years from 2022. M4 adds two groups of terms to M3: PV capacity times solar exposure, and the two temperatures (one straight line each, for the whole year). To see which group is responsible, M4 is refitted and replayed twice, each time with one group switched off. Nothing about M4 itself changes, and the two variants are not scored with the models.

The first table gives POE50 of the annual daytime low by test year: the actual, M3, M4 and the two variants, with the mean error (actual minus POE50) in the last row. The variant that stays close to M4 still holds the terms that matter. The second table gives the mean pinball loss of the same four on both reported targets.

In [8]:
variants = evaluate.m4_variant_forecasts(total, pv, weather, cfg)
compared = pd.concat([forecasts[forecasts["model"].isin(["M3", "M4"])], variants])
compared = compared[compared["target"].isin(["daytime_low", "minimum"])]

rows = compared[(compared["target"] == "daytime_low") & (compared["level"] == "annual")]
m4_terms = rows.pivot(index="year", columns="model", values="poe50")
m4_terms.insert(0, "actual", rows.groupby("year")["actual"].first())
m4_terms.to_csv(checks / "check_m4_terms.csv")

# Mean error of POE50 by column: actual minus forecast, so positive means forecasts too low.
forecast_columns = m4_terms.drop(columns="actual")
mean_error = forecast_columns.rsub(m4_terms["actual"], axis=0).mean().rename("mean error")
pd.concat([m4_terms, mean_error.to_frame().T])

model,actual,M3,M4,M4 without PV x solar,M4 without temperature
2019,"1,759","1,782","1,721","1,733","1,771"
2020,"1,671","1,639","1,576","1,600","1,621"
2021,"1,357","1,426","1,307","1,353","1,394"
2022,"1,458","1,110",999,"1,052","1,054"
2023,"1,080","1,016",921,"1,029",871
2024,"1,019",875,757,843,760
2025,880,839,683,760,707
mean error,NaN,77,180,122,150


In [9]:
pd.set_option("display.float_format", "{:,.1f}".format)
evaluate.score(compared, ["target", "level", "model"])["pinball_mean"].unstack("model")

model                 M3   M4  M4 without PV x solar  M4 without temperature
target      level                                                           
daytime_low annual  43.3 63.5                   45.7                    62.6
            monthly 49.6 51.8                   51.5                    51.1
minimum     annual  41.6 60.0                   43.6                    60.7
            monthly 41.4 44.1                   42.9                    44.1

The last table describes M4's weather terms themselves, for the daytime low, at each origin. The PV x solar term is a product, so its pull grows with PV capacity, and the sunniest days of a target year carry a larger product than any day the model was fitted on.

For each test year the table gives the coefficient on PV x solar (MW per 1,000 MW x MJ/m2) and on maximum and minimum temperature (MW per degree C); the largest PV x solar among the training days; PV x solar on the sunniest days of the target year (its 99th percentile under the year's own weather); and how far, in per cent, the second lies beyond the first.

In [10]:
m4_coefficients = evaluate.m4_weather_terms(total, pv, weather, cfg)
m4_coefficients.to_csv(checks / "check_m4_coefficients.csv")
m4_coefficients

,pv_solar_mw_per_1000,tmax_mw_per_degree,tmin_mw_per_degree,pv_solar_max_in_training,pv_solar_high_in_target,beyond_training_pct
year,,,,,,
2019,-18.2,11.6,26.0,"17,143.7","21,472.6",25.3
2020,-14.5,9.9,27.7,"22,012.3","28,451.4",29.3
2021,-13.3,9.4,30.0,"28,956.9","35,906.1",24.0
2022,-14.5,8.6,29.8,"37,498.6","44,554.8",18.8
2023,-14.6,6.9,29.6,"45,744.7","52,352.1",14.4
2024,-13.9,6.2,32.4,"54,541.4","63,185.4",15.8
2025,-15.5,5.8,34.3,"63,581.8","68,550.2",7.8


## 8. Result

The forecasts are saved for notebooks 06 and 07. M5 and M6 are plain XGBoost.

In [11]:
pd.set_option("display.float_format", "{:,.0f}".format)
print(f"Test years: {first_test} to {last_year}; models: {sorted(forecasts['model'].unique())}")
print(f"Simulated years behind each M2 to M6 forecast: "
      f"{forecasts.loc[forecasts['model'] != 'M1', 'n_pool'].min():,.0f} to "
      f"{forecasts['n_pool'].max():,.0f}")
print("Mean error of POE50 on the annual daytime low (actual minus forecast, MW):")
errors = daytime.assign(error=daytime["actual"] - daytime["poe50"])
print(errors.groupby("model")["error"].mean().round().to_dict())

Test years: 2019 to 2025; models: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']
Simulated years behind each M2 to M6 forecast: 1,700 to 2,300
Mean error of POE50 on the annual daytime low (actual minus forecast, MW):
{'M1': -126.0, 'M2': -62.0, 'M3': 77.0, 'M4': 180.0, 'M5': -25.0, 'M6': -49.0}
